# Tuned 2048 DQN Notebook (Reward Shaping + N-step + Prioritized Replay)

Notebook này là bản nâng cấp từ `DQN_for_2048_game.ipynb`, đã tích hợp các thay đổi được đề xuất để cải thiện:

- **Observation wrapper tốt hơn**: one-hot 4x4x16 + scalar feature maps
- **CNN + dueling heads**
- **Double DQN**
- **Huber loss**
- **Reward shaping mạnh hơn cho 2048**:
  - empty cells
  - monotonicity
  - smoothness
  - max tile progress
  - giữ max tile ở corner
- **N-step targets**
- **Prioritized replay**
- **Đánh giá multi-seed**
- **In score từng eval seed trước khi in thống kê tổng**
- **Metric “fraction of illegal actions avoided”** được đo theo **raw argmax trước khi mask**, nên có ý nghĩa hơn

> Lưu ý quan trọng:
> - Agent có thể **train bằng reward shaping**, nhưng phần **`mean greedy return` trong evaluation luôn là game score thật** (tổng delta-return thật của game 2048), không phải shaped reward.
> - Nếu chỉ muốn debug nhanh, hãy giảm `NUM_EPISODES`. Nếu muốn tối ưu kết quả cuối, thường phải train lâu hơn.

In [1]:

# Colab / notebook setup
!python -V
!pip -q install --upgrade pip
!pip -q install open-spiel torch matplotlib pandas imageio tqdm


Python 3.12.13


In [2]:

import copy
import json
import math
import os
import random
import re
import time
import traceback
from collections import Counter, deque, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pyspiel

print("PyTorch version:", torch.__version__)
print("OpenSpiel version:", pyspiel.__version__ if hasattr(pyspiel, "__version__") else "unknown")
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


/workspace/xla-dev/tt-xla/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.1+cpu
OpenSpiel version: 1.6.12
CUDA available: False


device(type='cpu')


## 1. Game inspection and robust helpers

OpenSpiel 2048 có chance nodes, nên wrapper sẽ auto-resolve các node ngẫu nhiên. Phần helper dưới đây cũng cố tình **robust** để notebook dễ chạy trên Colab hơn.


In [3]:
game = pyspiel.load_game("2048")
state = game.new_initial_state()

print("Registered game:", game.get_type().short_name)
print("Num distinct actions:", game.num_distinct_actions())
print("Observation tensor shape:", game.observation_tensor_shape())
print("Observation tensor size:", game.observation_tensor_size())
print("Max game length:", game.max_game_length())
print("Initial state:\n", state)

Registered game: 2048
Num distinct actions: 4
Observation tensor shape: [4, 4]
Observation tensor size: 16
Max game length: 8192
Initial state:
     0    0    0    0
    0    0    0    0
    0    0    0    0
    0    0    0    0



In [4]:
def extract_obs_grid(state, player_id=0):
    """Return a 4x4 float32 board from OpenSpiel observation/state."""
    for fn_name, args in [
        ("observation_tensor", (player_id,)),
        ("observation_tensor", tuple()),
        ("information_state_tensor", (player_id,)),
        ("information_state_tensor", tuple()),
    ]:
        fn = getattr(state, fn_name, None)
        if fn is None:
            continue
        try:
            obs = fn(*args)
            obs = np.asarray(obs, dtype=np.float32)
            if obs.shape == (4, 4):
                return obs
            if obs.size == 16:
                return obs.reshape(4, 4)
        except TypeError:
            pass

    numbers = [int(x) for x in re.findall(r"-?\d+", str(state))]
    if len(numbers) >= 16:
        return np.asarray(numbers[:16], dtype=np.float32).reshape(4, 4)

    raise RuntimeError("Could not extract a 4x4 board from the OpenSpiel state.")


def get_state_return(state):
    returns_fn = getattr(state, "returns", None)
    if returns_fn is None:
        return 0.0
    try:
        vals = returns_fn()
    except TypeError:
        vals = returns_fn
    vals = np.asarray(vals, dtype=np.float32).reshape(-1)
    return float(vals[0]) if vals.size else 0.0


def get_state_reward(state):
    rewards_fn = getattr(state, "rewards", None)
    if rewards_fn is None:
        return 0.0
    try:
        vals = rewards_fn()
    except TypeError:
        vals = rewards_fn
    vals = np.asarray(vals, dtype=np.float32).reshape(-1)
    return float(vals[0]) if vals.size else 0.0


def auto_resolve_chance_nodes(state, rng):
    while state.is_chance_node():
        outcomes = state.chance_outcomes()
        actions = np.asarray([a for a, _ in outcomes], dtype=np.int64)
        probs = np.asarray([p for _, p in outcomes], dtype=np.float64)
        probs = probs / probs.sum()
        chosen = int(rng.choice(actions, p=probs))
        state.apply_action(chosen)


def board_to_log2(board):
    board = np.asarray(board, dtype=np.float32)
    out = np.zeros_like(board, dtype=np.int64)
    mask = board > 0
    out[mask] = np.log2(board[mask]).astype(np.int64)
    return out


def monotonicity_score(log_board):
    log_board = np.asarray(log_board, dtype=np.int64)
    total = 0
    for arr in list(log_board) + list(log_board.T):
        arr = np.asarray(arr)
        non_inc = np.sum(arr[:-1] >= arr[1:])
        non_dec = np.sum(arr[:-1] <= arr[1:])
        total += max(int(non_inc), int(non_dec))
    return float(total) / 24.0  # 8 lines * 3 adjacent comparisons


def smoothness_score(log_board):
    log_board = np.asarray(log_board, dtype=np.int64)
    diffs = []
    for r in range(4):
        for c in range(4):
            if log_board[r, c] == 0:
                continue
            if r + 1 < 4 and log_board[r + 1, c] > 0:
                diffs.append(abs(int(log_board[r, c]) - int(log_board[r + 1, c])))
            if c + 1 < 4 and log_board[r, c + 1] > 0:
                diffs.append(abs(int(log_board[r, c]) - int(log_board[r, c + 1])))
    if not diffs:
        return 1.0
    return float(np.clip(1.0 - np.mean(diffs) / 15.0, 0.0, 1.0))


def compute_board_metrics(board, max_log2_channel=15):
    board = np.asarray(board, dtype=np.float32).reshape(4, 4)
    log_board = board_to_log2(board)
    clipped = np.clip(log_board, 0, max_log2_channel)

    empty_count = int(np.sum(board == 0))
    empty_frac = float(empty_count / 16.0)
    max_tile = int(np.max(board))
    max_log = float(np.max(clipped))
    corners = [board[0, 0], board[0, 3], board[3, 0], board[3, 3]]
    corner_flag = float(max_tile > 0 and max_tile in corners)

    return {
        "board": board,
        "log_board": clipped,
        "empty_count": empty_count,
        "empty_frac": empty_frac,
        "max_tile": max_tile,
        "max_log": max_log,
        "corner_flag": corner_flag,
        "monotonicity": monotonicity_score(clipped),
        "smoothness": smoothness_score(clipped),
    }


def encode_board_features(board, max_log2_channel=15):
    """
    Encode board as:
      - 16 one-hot channels for values {empty, 2, 4, ..., 32768}
      - 5 scalar feature maps: empty_count, max_tile_log2, corner_flag, monotonicity, smoothness
    Output shape: [21, 4, 4]
    """
    metrics = compute_board_metrics(board, max_log2_channel=max_log2_channel)
    clipped = metrics["log_board"]

    one_hot = np.eye(max_log2_channel + 1, dtype=np.float32)[clipped]      # [4, 4, 16]
    one_hot = np.transpose(one_hot, (2, 0, 1))                             # [16, 4, 4]

    scalars = [
        metrics["empty_frac"],
        float(metrics["max_log"] / max_log2_channel) if max_log2_channel > 0 else 0.0,
        metrics["corner_flag"],
        metrics["monotonicity"],
        metrics["smoothness"],
    ]
    scalar_maps = np.stack([
        np.full((4, 4), fill_value=v, dtype=np.float32) for v in scalars
    ], axis=0)

    return np.concatenate([one_hot, scalar_maps], axis=0).astype(np.float32)


def make_legal_mask(num_actions, legal_actions_list):
    mask = np.zeros(num_actions, dtype=np.float32)
    if legal_actions_list:
        mask[np.asarray(legal_actions_list, dtype=np.int64)] = 1.0
    return mask

## 2. Tuned OpenSpiel 2048 wrapper

Wrapper này hỗ trợ nhiều chế độ reward:

- `delta_return`: chỉ dùng delta score thật của game
- `state_rewards`: dùng `state.rewards()`
- `shaped`: delta score + shaping cơ bản
- `strategic_shaped`: **shaping mạnh hơn cho 2048**, kết hợp:
  - merge reward ở thang log
  - empty cells
  - tiến bộ max tile
  - monotonicity
  - smoothness
  - giữ max tile ở corner

`strategic_shaped` thường hữu ích hơn khi mục tiêu là kéo **max tile** và **mean greedy return** lên, nhưng evaluation cuối vẫn sẽ báo **game score thật**.

In [5]:
class OpenSpiel2048FeatureEnv:
    def __init__(
        self,
        seed=42,
        reward_mode="strategic_shaped",
        empty_cell_bonus=0.5,
        new_max_tile_bonus=6.0,
        monotonicity_bonus=2.0,
        smoothness_bonus=1.0,
        corner_bonus=2.0,
        keep_max_in_corner_bonus=0.5,
        merge_log_reward_scale=4.0,
    ):
        self.game = pyspiel.load_game("2048")
        self.player_id = 0
        self.num_actions = self.game.num_distinct_actions()
        self.rng = np.random.default_rng(seed)
        self.reward_mode = reward_mode
        self.empty_cell_bonus = float(empty_cell_bonus)
        self.new_max_tile_bonus = float(new_max_tile_bonus)
        self.monotonicity_bonus = float(monotonicity_bonus)
        self.smoothness_bonus = float(smoothness_bonus)
        self.corner_bonus = float(corner_bonus)
        self.keep_max_in_corner_bonus = float(keep_max_in_corner_bonus)
        self.merge_log_reward_scale = float(merge_log_reward_scale)
        self.state = None

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.game.new_initial_state()
        auto_resolve_chance_nodes(self.state, self.rng)
        board = extract_obs_grid(self.state, self.player_id)
        return encode_board_features(board)

    def legal_actions(self):
        if self.state is None or self.state.is_terminal():
            return []
        return list(self.state.legal_actions())

    def step(self, action):
        if self.state is None:
            raise RuntimeError("Call reset() before step().")
        legal = self.legal_actions()
        if action not in legal:
            raise ValueError(f"Illegal action {action}. Legal actions are {legal}")

        prev_board = extract_obs_grid(self.state, self.player_id)
        prev_metrics = compute_board_metrics(prev_board)
        prev_return = get_state_return(self.state)

        self.state.apply_action(int(action))
        auto_resolve_chance_nodes(self.state, self.rng)

        board = extract_obs_grid(self.state, self.player_id)
        done = self.state.is_terminal()
        metrics = compute_board_metrics(board)

        total_return = get_state_return(self.state)
        delta_return = float(total_return - prev_return)
        raw_state_reward = get_state_reward(self.state)
        delta_empty = int(metrics["empty_count"] - prev_metrics["empty_count"])
        delta_max_log = float(max(0.0, metrics["max_log"] - prev_metrics["max_log"]))
        delta_monotonicity = float(metrics["monotonicity"] - prev_metrics["monotonicity"])
        delta_smoothness = float(metrics["smoothness"] - prev_metrics["smoothness"])
        delta_corner_flag = float(metrics["corner_flag"] - prev_metrics["corner_flag"])
        new_max_tile = float(metrics["max_tile"] > prev_metrics["max_tile"])
        merge_log_reward = math.log2(1.0 + max(delta_return, 0.0))

        if self.reward_mode == "delta_return":
            reward = delta_return
        elif self.reward_mode == "state_rewards":
            reward = raw_state_reward
        elif self.reward_mode == "shaped":
            reward = (
                delta_return
                + self.empty_cell_bonus * delta_empty
                + self.new_max_tile_bonus * new_max_tile
            )
        elif self.reward_mode == "strategic_shaped":
            reward = (
                self.merge_log_reward_scale * merge_log_reward
                + self.empty_cell_bonus * delta_empty
                + self.new_max_tile_bonus * delta_max_log
                + self.monotonicity_bonus * delta_monotonicity
                + self.smoothness_bonus * delta_smoothness
                + self.corner_bonus * delta_corner_flag
                + self.keep_max_in_corner_bonus * metrics["corner_flag"]
            )
        else:
            raise ValueError(f"Unknown reward_mode: {self.reward_mode}")

        info = {
            "board": board.astype(np.int64),
            "state_text": str(self.state),
            "legal_actions": self.legal_actions(),
            "delta_return": delta_return,
            "raw_state_reward": raw_state_reward,
            "delta_empty": delta_empty,
            "delta_max_log": delta_max_log,
            "delta_monotonicity": delta_monotonicity,
            "delta_smoothness": delta_smoothness,
            "delta_corner_flag": delta_corner_flag,
            "new_max_tile_increased": bool(new_max_tile),
            "merge_log_reward": merge_log_reward,
            "max_tile": int(metrics["max_tile"]),
            "corner_flag": bool(metrics["corner_flag"]),
            "monotonicity": float(metrics["monotonicity"]),
            "smoothness": float(metrics["smoothness"]),
            "total_return": total_return,
        }
        return encode_board_features(board), float(reward), done, info

    def render(self):
        if self.state is None:
            print("<empty env>")
        else:
            print(self.state)

In [6]:

# Quick sanity check
sanity_env = OpenSpiel2048FeatureEnv(seed=123, reward_mode="delta_return")
obs = sanity_env.reset(seed=123)
print("Encoded obs shape:", obs.shape)
print("Initial legal actions:", sanity_env.legal_actions())
sanity_env.render()


Encoded obs shape: (21, 4, 4)
Initial legal actions: [0, 1, 2, 3]
    2    0    0    0
    0    0    0    0
    0    0    4    0
    0    0    0    0



## 3. Replay buffer, prioritized replay, n-step returns, and tuned Q-network

Phần này thêm hai nâng cấp quan trọng để kéo **max tile** và **mean greedy return**:

- **N-step targets**: tín hiệu dài hạn lan ngược nhanh hơn
- **Prioritized replay**: ưu tiên sample transition có TD error lớn hơn

Kết hợp với **Double DQN** + **Huber loss**, đây là bộ thay đổi hợp lý nhất nếu agent đang bị kẹt ở mức 256/512.

In [7]:
Transition = namedtuple(
    "Transition",
    ["obs", "action", "reward", "next_obs", "done", "legal_mask", "next_legal_mask", "discount"],
)


class UniformReplayBuffer:
    def __init__(self, capacity):
        self.capacity = int(capacity)
        self.buffer = deque(maxlen=self.capacity)

    def __len__(self):
        return len(self.buffer)

    def add(self, *args, priority=None):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size, beta=None):
        idx = np.random.choice(len(self.buffer), size=batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        weights = np.ones(batch_size, dtype=np.float32)
        return Transition(*zip(*batch)), idx, weights

    def update_priorities(self, indices, priorities):
        return None


class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, priority_eps=1e-6):
        self.capacity = int(capacity)
        self.alpha = float(alpha)
        self.priority_eps = float(priority_eps)
        self.buffer = []
        self.priorities = np.zeros(self.capacity, dtype=np.float32)
        self.pos = 0

    def __len__(self):
        return len(self.buffer)

    def add(self, *args, priority=None):
        transition = Transition(*args)
        max_prio = float(self.priorities[:len(self.buffer)].max()) if self.buffer else 1.0
        prio = max(float(priority) if priority is not None else max_prio, self.priority_eps)

        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition

        self.priorities[self.pos] = prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        n = len(self.buffer)
        probs = self.priorities[:n] ** self.alpha
        probs_sum = probs.sum()
        if probs_sum <= 0:
            probs = np.full(n, 1.0 / n, dtype=np.float32)
        else:
            probs = probs / probs_sum

        replace = n < batch_size
        indices = np.random.choice(n, size=batch_size, replace=replace, p=probs)
        batch = [self.buffer[i] for i in indices]

        weights = (n * probs[indices]) ** (-float(beta))
        weights = weights / max(weights.max(), 1e-8)
        weights = weights.astype(np.float32)

        return Transition(*zip(*batch)), indices, weights

    def update_priorities(self, indices, priorities):
        for idx, prio in zip(indices, priorities):
            self.priorities[int(idx)] = max(float(abs(prio)), self.priority_eps)


class NStepTransitionAccumulator:
    def __init__(self, n_step, gamma):
        self.n_step = int(max(1, n_step))
        self.gamma = float(gamma)
        self.buffer = deque()

    def reset(self):
        self.buffer.clear()

    def _build_one(self):
        reward = 0.0
        discount = 1.0
        done = False
        next_obs = None
        next_legal_mask = None

        for i, item in enumerate(self.buffer):
            reward += discount * float(item["reward"])
            discount *= self.gamma
            next_obs = item["next_obs"]
            next_legal_mask = item["next_legal_mask"]
            done = bool(item["done"])
            if done or (i + 1) >= self.n_step:
                break

        first = self.buffer[0]
        self.buffer.popleft()

        return Transition(
            first["obs"],
            int(first["action"]),
            float(reward),
            next_obs,
            bool(done),
            first["legal_mask"],
            next_legal_mask,
            float(discount),
        )

    def add(self, obs, action, reward, next_obs, done, legal_mask, next_legal_mask):
        self.buffer.append({
            "obs": obs,
            "action": action,
            "reward": reward,
            "next_obs": next_obs,
            "done": done,
            "legal_mask": legal_mask,
            "next_legal_mask": next_legal_mask,
        })

        ready = []
        if len(self.buffer) >= self.n_step:
            ready.append(self._build_one())

        if done:
            while self.buffer:
                ready.append(self._build_one())

        return ready


class DuelingConvQNet(nn.Module):
    def __init__(self, in_channels, num_actions, dueling=True):
        super().__init__()
        self.dueling = dueling
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.trunk = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
        )
        if dueling:
            self.value_stream = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, 1),
            )
            self.adv_stream = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, num_actions),
            )
        else:
            self.head = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, num_actions),
            )

    def forward(self, x):
        z = self.features(x)
        z = self.trunk(z)
        if self.dueling:
            value = self.value_stream(z)
            adv = self.adv_stream(z)
            return value + adv - adv.mean(dim=1, keepdim=True)
        return self.head(z)


## 4. Central config

Các config này là bản “tuned” theo hướng đã đề xuất. Nếu muốn debug nhanh, giảm `NUM_EPISODES` xuống trước khi chạy full.


In [8]:

CONFIG = {
    # Repro / training
    "TRAIN_SEED": 7,
    "NUM_EPISODES": 3000,          # debug: 800-1500, serious run: 4000-8000 if time budget allows
    "BUFFER_SIZE": 100_000,
    "BATCH_SIZE": 128,
    "GAMMA": 0.99,
    "LR": 2.5e-4,
    "TARGET_SYNC_EVERY": 1000,
    "LEARN_START": 5000,
    "LEARN_EVERY": 4,
    "MAX_STEPS_PER_EPISODE": 5000,
    "GRAD_CLIP": 10.0,

    # Exploration
    "EPS_START": 1.0,
    "EPS_END": 0.01,
    "EPS_DECAY_STEPS": 100_000,

    # Architecture / algorithm
    "USE_DOUBLE_DQN": True,
    "USE_DUELING": True,
    "N_STEP": 3,

    # Replay
    "USE_PRIORITIZED_REPLAY": True,
    "PER_ALPHA": 0.6,
    "PER_BETA_START": 0.4,
    "PER_BETA_END": 1.0,
    "PER_BETA_ANNEAL_STEPS": 150_000,
    "PRIORITY_EPS": 1e-6,

    # Reward mode
    "REWARD_MODE": "strategic_shaped",   # choose: delta_return, state_rewards, shaped, strategic_shaped
    "EMPTY_CELL_BONUS": 0.5,
    "NEW_MAX_TILE_BONUS": 6.0,
    "MONOTONICITY_BONUS": 2.0,
    "SMOOTHNESS_BONUS": 1.0,
    "CORNER_BONUS": 2.0,
    "KEEP_MAX_IN_CORNER_BONUS": 0.5,
    "MERGE_LOG_REWARD_SCALE": 4.0,

    # Probing during training
    "PROBE_EVERY_EPISODES": 100,
    "PROBE_SEEDS": [1001, 1002, 1003, 1004, 1005],

    # Final evaluation
    "EVAL_SEEDS": list(range(2001, 2051)),
    "PRINT_EACH_EVAL_SEED": True,

    # Save / resume / hang protection
    "SAVE_PATH": "dqn_2048_tuned_cnn_double_dueling_per_nstep.pt",
    "CHECKPOINT_DIR": "./ckpt_2048_safe",
    "CHECKPOINT_EVERY_EPISODES": 50,
    "HEARTBEAT_EVERY_EPISODES": 10,
    "PRINT_HEARTBEAT": True,
    "EPISODE_WARN_SECONDS": 60.0,
    "EPISODE_ABORT_SECONDS": 180.0,
    "AUTO_RESUME": True,
    "RESUME_PATH": None,                # None -> use latest checkpoint in CHECKPOINT_DIR if AUTO_RESUME=True
    "SAVE_REPLAY_IN_CHECKPOINT": False, # False by default because replay can get very large and slow to save
    "STATUS_JSON": "train_status_2048.json",
}

CONFIG


{'TRAIN_SEED': 7,
 'NUM_EPISODES': 3000,
 'BUFFER_SIZE': 100000,
 'BATCH_SIZE': 128,
 'GAMMA': 0.99,
 'LR': 0.00025,
 'TARGET_SYNC_EVERY': 1000,
 'LEARN_START': 5000,
 'LEARN_EVERY': 4,
 'MAX_STEPS_PER_EPISODE': 5000,
 'GRAD_CLIP': 10.0,
 'EPS_START': 1.0,
 'EPS_END': 0.01,
 'EPS_DECAY_STEPS': 100000,
 'USE_DOUBLE_DQN': True,
 'USE_DUELING': True,
 'N_STEP': 3,
 'USE_PRIORITIZED_REPLAY': True,
 'PER_ALPHA': 0.6,
 'PER_BETA_START': 0.4,
 'PER_BETA_END': 1.0,
 'PER_BETA_ANNEAL_STEPS': 150000,
 'PRIORITY_EPS': 1e-06,
 'REWARD_MODE': 'strategic_shaped',
 'EMPTY_CELL_BONUS': 0.5,
 'NEW_MAX_TILE_BONUS': 6.0,
 'MONOTONICITY_BONUS': 2.0,
 'SMOOTHNESS_BONUS': 1.0,
 'CORNER_BONUS': 2.0,
 'KEEP_MAX_IN_CORNER_BONUS': 0.5,
 'MERGE_LOG_REWARD_SCALE': 4.0,
 'PROBE_EVERY_EPISODES': 100,
 'PROBE_SEEDS': [1001, 1002, 1003, 1004, 1005],
 'EVAL_SEEDS': [2001,
  2002,
  2003,
  2004,
  2005,
  2006,
  2007,
  2008,
  2009,
  2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  20


## 6b. Safe checkpoint / resume utilities

Cell này giúp notebook **không mất trắng** khi train lâu:
- lưu checkpoint định kỳ
- tự resume từ checkpoint gần nhất
- in heartbeat để biết notebook vẫn đang chạy
- cảnh báo / cắt episode nếu chạy quá lâu
- tự lưu checkpoint khi `KeyboardInterrupt` hoặc gặp lỗi


In [9]:

def ensure_dir(path_like):
    path = Path(path_like)
    path.mkdir(parents=True, exist_ok=True)
    return path


def checkpoint_dir(cfg):
    return ensure_dir(cfg["CHECKPOINT_DIR"])


def checkpoint_path(cfg, train_seed, episode, tag="episode"):
    ckpt_dir = checkpoint_dir(cfg)
    return ckpt_dir / f"seed_{int(train_seed)}_{tag}_ep_{int(episode):05d}.pt"


def latest_checkpoint_path(cfg, train_seed=None):
    ckpt_dir = checkpoint_dir(cfg)
    if train_seed is None:
        matches = sorted(ckpt_dir.glob("seed_*_latest.pt"))
    else:
        matches = sorted(ckpt_dir.glob(f"seed_{int(train_seed)}_latest.pt"))
    return matches[-1] if matches else None


def _rng_state_payload():
    payload = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_random_state": torch.get_rng_state(),
        "cuda_random_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    return payload


def _restore_rng_state(payload):
    if not payload:
        return
    try:
        if payload.get("python_random_state") is not None:
            random.setstate(payload["python_random_state"])
        if payload.get("numpy_random_state") is not None:
            np.random.set_state(payload["numpy_random_state"])
        if payload.get("torch_random_state") is not None:
            torch.set_rng_state(payload["torch_random_state"])
        if torch.cuda.is_available() and payload.get("cuda_random_state_all") is not None:
            torch.cuda.set_rng_state_all(payload["cuda_random_state_all"])
    except Exception as exc:
        print(f"[warn] Could not fully restore RNG state: {exc}")


def write_status_json(cfg, payload):
    status_path = Path(cfg["STATUS_JSON"])
    status = {
        "saved_at": payload.get("saved_at"),
        "train_seed": payload.get("train_seed"),
        "episode_completed": payload.get("episode_completed"),
        "global_step": payload.get("global_step"),
        "checkpoint_path": str(payload.get("checkpoint_path", "")),
        "stop_reason": payload.get("stop_reason", "running"),
    }
    status_path.write_text(json.dumps(status, indent=2))
    return status_path


def make_training_checkpoint(
    cfg,
    train_seed,
    episode_completed,
    global_step,
    q_net,
    target_net,
    optimizer,
    replay,
    episode_train_rewards,
    episode_true_scores,
    episode_lengths,
    episode_max_tiles,
    loss_history,
    probe_records,
    in_channels,
    num_actions,
    stop_reason="running",
    include_replay=None,
):
    if include_replay is None:
        include_replay = bool(cfg.get("SAVE_REPLAY_IN_CHECKPOINT", False))
    payload = {
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "train_seed": int(train_seed),
        "episode_completed": int(episode_completed),
        "global_step": int(global_step),
        "config": copy.deepcopy(cfg),
        "in_channels": int(in_channels),
        "num_actions": int(num_actions),
        "model_state_dict": q_net.state_dict(),
        "target_state_dict": target_net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "episode_returns": list(episode_train_rewards),
        "episode_true_scores": list(episode_true_scores),
        "episode_lengths": list(episode_lengths),
        "episode_max_tiles": list(episode_max_tiles),
        "loss_history": list(loss_history),
        "probe_records": list(probe_records),
        "stop_reason": stop_reason,
        "rng_state": _rng_state_payload(),
    }
    if include_replay:
        payload["replay"] = replay
        payload["replay_saved"] = True
    else:
        payload["replay_saved"] = False
    return payload


def save_training_checkpoint(payload, cfg, train_seed, episode_completed, tag="episode"):
    path = checkpoint_path(cfg, train_seed, episode_completed, tag=tag)
    payload = dict(payload)
    payload["checkpoint_path"] = str(path)
    torch.save(payload, path)
    latest_path = checkpoint_dir(cfg) / f"seed_{int(train_seed)}_latest.pt"
    torch.save(payload, latest_path)
    write_status_json(cfg, payload)
    return path, latest_path


def load_training_checkpoint(path, map_location=DEVICE):
    path = Path(path)
    payload = torch.load(path, map_location=map_location, weights_only=False)
    return payload


## 5. Action selection, Double DQN update, PER weights, and evaluation helpers

Điểm mới trong phần này:

- **Double DQN** target
- **importance sampling weights** cho prioritized replay
- **n-step discount** trong target
- evaluation in ra **score từng seed**
- `mean_greedy_return` luôn là **game score thật**, kể cả khi train bằng reward shaping

In [10]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def epsilon_by_step(step, cfg):
    frac = min(1.0, step / cfg["EPS_DECAY_STEPS"])
    return cfg["EPS_START"] + frac * (cfg["EPS_END"] - cfg["EPS_START"])


def per_beta_by_step(step, cfg):
    frac = min(1.0, step / cfg["PER_BETA_ANNEAL_STEPS"])
    return cfg["PER_BETA_START"] + frac * (cfg["PER_BETA_END"] - cfg["PER_BETA_START"])


def masked_greedy_action(q_net, obs, legal_actions, num_actions, epsilon=0.0, device=DEVICE, return_debug=False):
    if len(legal_actions) == 0:
        raise RuntimeError("No legal actions available.")

    with torch.no_grad():
        x = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        q_values = q_net(x).squeeze(0).detach().cpu().numpy()

    raw_argmax = int(np.argmax(q_values))
    raw_argmax_is_legal = raw_argmax in set(legal_actions)

    if random.random() < epsilon:
        action = int(random.choice(legal_actions))
    else:
        masked_q = q_values.copy()
        illegal = np.ones(num_actions, dtype=bool)
        illegal[np.asarray(legal_actions, dtype=np.int64)] = False
        masked_q[illegal] = -1e9
        action = int(np.argmax(masked_q))

    if return_debug:
        return action, {
            "raw_q": q_values,
            "raw_argmax": raw_argmax,
            "raw_argmax_is_legal": raw_argmax_is_legal,
        }
    return action


def dqn_update(batch, weights, q_net, target_net, optimizer, cfg):
    obs = torch.tensor(np.asarray(batch.obs), dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(batch.action, dtype=torch.int64, device=DEVICE).unsqueeze(1)
    rewards = torch.tensor(batch.reward, dtype=torch.float32, device=DEVICE)
    next_obs = torch.tensor(np.asarray(batch.next_obs), dtype=torch.float32, device=DEVICE)
    dones = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE)
    discounts = torch.tensor(batch.discount, dtype=torch.float32, device=DEVICE)
    next_legal_mask = torch.tensor(np.asarray(batch.next_legal_mask), dtype=torch.bool, device=DEVICE)
    weights_t = torch.tensor(np.asarray(weights), dtype=torch.float32, device=DEVICE)

    q_values = q_net(obs)
    q_sa = q_values.gather(1, actions).squeeze(1)

    with torch.no_grad():
        no_legal_next = ~next_legal_mask.any(dim=1)

        if cfg["USE_DOUBLE_DQN"]:
            online_next_q = q_net(next_obs).masked_fill(~next_legal_mask, -1e9)
            next_actions = online_next_q.argmax(dim=1, keepdim=True)
            target_next_q = target_net(next_obs).gather(1, next_actions).squeeze(1)
            target_next_q = target_next_q.masked_fill(no_legal_next, 0.0)
        else:
            target_next_q = target_net(next_obs).masked_fill(~next_legal_mask, -1e9)
            target_next_q = target_next_q.max(dim=1).values
            target_next_q = target_next_q.masked_fill(no_legal_next, 0.0)

        target = rewards + (1.0 - dones) * discounts * target_next_q

    td_error = target - q_sa
    per_item_loss = F.smooth_l1_loss(q_sa, target, reduction="none")
    loss = (weights_t * per_item_loss).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), cfg["GRAD_CLIP"])
    optimizer.step()

    return float(loss.item()), np.abs(td_error.detach().cpu().numpy()) + cfg["PRIORITY_EPS"]


def evaluate_greedy_policy(
    q_net,
    seeds,
    cfg,
    reward_mode=None,
    render_rollout_seed=None,
    print_each_seed=False,
):
    reward_mode = reward_mode or cfg["REWARD_MODE"]
    per_seed = []
    tile_counter = Counter()
    raw_argmax_legal_steps = 0
    total_decision_steps = 0
    rollout_trace = None

    for seed in seeds:
        env = OpenSpiel2048FeatureEnv(
            seed=seed,
            reward_mode=reward_mode,
            empty_cell_bonus=cfg["EMPTY_CELL_BONUS"],
            new_max_tile_bonus=cfg["NEW_MAX_TILE_BONUS"],
            monotonicity_bonus=cfg["MONOTONICITY_BONUS"],
            smoothness_bonus=cfg["SMOOTHNESS_BONUS"],
            corner_bonus=cfg["CORNER_BONUS"],
            keep_max_in_corner_bonus=cfg["KEEP_MAX_IN_CORNER_BONUS"],
            merge_log_reward_scale=cfg["MERGE_LOG_REWARD_SCALE"],
        )
        obs = env.reset(seed=seed)
        done = False
        total_train_reward = 0.0
        total_game_score = 0.0
        steps = 0
        max_tile = 0
        local_trace = []

        while not done and steps < cfg["MAX_STEPS_PER_EPISODE"]:
            legal = env.legal_actions()
            action, dbg = masked_greedy_action(
                q_net,
                obs,
                legal,
                env.num_actions,
                epsilon=0.0,
                return_debug=True,
                device=DEVICE,
            )
            raw_argmax_legal_steps += int(dbg["raw_argmax_is_legal"])
            total_decision_steps += 1

            next_obs, reward, done, info = env.step(action)
            total_train_reward += reward
            total_game_score += float(info["delta_return"])
            steps += 1
            max_tile = max(max_tile, int(info["max_tile"]))

            if render_rollout_seed is not None and seed == render_rollout_seed:
                local_trace.append({
                    "action": int(action),
                    "reward": float(reward),
                    "game_delta_return": float(info["delta_return"]),
                    "board": info["board"].copy(),
                    "state_text": info["state_text"],
                    "legal_actions": list(legal),
                    "raw_argmax": int(dbg["raw_argmax"]),
                    "raw_argmax_is_legal": bool(dbg["raw_argmax_is_legal"]),
                })

            obs = next_obs

        row = {
            "seed": int(seed),
            "greedy_return": float(total_game_score),      # true game score
            "train_reward_sum": float(total_train_reward), # reward under the chosen reward wrapper
            "steps": int(steps),
            "max_tile": int(max_tile),
        }
        per_seed.append(row)
        tile_counter[int(max_tile)] += 1
        if print_each_seed:
            print(
                f"Eval seed {seed}: score={row['greedy_return']:.2f}, "
                f"max_tile={row['max_tile']}, steps={row['steps']}, "
                f"train_reward={row['train_reward_sum']:.2f}"
            )
        if render_rollout_seed is not None and seed == render_rollout_seed:
            rollout_trace = local_trace

    df = pd.DataFrame(per_seed)
    summary = {
        "mean_greedy_return": float(df["greedy_return"].mean()),
        "std_greedy_return": float(df["greedy_return"].std(ddof=0)),
        "median_greedy_return": float(df["greedy_return"].median()),
        "mean_train_reward": float(df["train_reward_sum"].mean()),
        "mean_max_tile": float(df["max_tile"].mean()),
        "max_tile_distribution": dict(sorted(tile_counter.items())),
        "fraction_illegal_actions_avoided": float(raw_argmax_legal_steps / max(total_decision_steps, 1)),
        "num_eval_seeds": int(len(seeds)),
        "table": df.sort_values(["max_tile", "greedy_return"], ascending=False).reset_index(drop=True),
        "per_seed_results": df.sort_values("seed").reset_index(drop=True),
    }
    return summary, rollout_trace

## 6. Training loop

Notebook này mặc định train **một seed**, nhưng evaluation cuối dùng **nhiều seed** để báo cáo metric ổn định hơn.

Điểm mới:
- dùng **n-step accumulator**
- có thể bật **prioritized replay**
- final evaluation sẽ **in score từng seed**

In [11]:

def train_one_seed(cfg, train_seed=None, resume_from=None):
    train_seed = cfg["TRAIN_SEED"] if train_seed is None else int(train_seed)
    set_all_seeds(train_seed)

    env = OpenSpiel2048FeatureEnv(
        seed=train_seed,
        reward_mode=cfg["REWARD_MODE"],
        empty_cell_bonus=cfg["EMPTY_CELL_BONUS"],
        new_max_tile_bonus=cfg["NEW_MAX_TILE_BONUS"],
        monotonicity_bonus=cfg["MONOTONICITY_BONUS"],
        smoothness_bonus=cfg["SMOOTHNESS_BONUS"],
        corner_bonus=cfg["CORNER_BONUS"],
        keep_max_in_corner_bonus=cfg["KEEP_MAX_IN_CORNER_BONUS"],
        merge_log_reward_scale=cfg["MERGE_LOG_REWARD_SCALE"],
    )

    sample_obs = env.reset(seed=train_seed)
    in_channels = int(sample_obs.shape[0])
    num_actions = int(env.num_actions)

    q_net = DuelingConvQNet(in_channels, num_actions, dueling=cfg["USE_DUELING"]).to(DEVICE)
    target_net = DuelingConvQNet(in_channels, num_actions, dueling=cfg["USE_DUELING"]).to(DEVICE)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=cfg["LR"])

    if cfg["USE_PRIORITIZED_REPLAY"]:
        replay = PrioritizedReplayBuffer(
            cfg["BUFFER_SIZE"],
            alpha=cfg["PER_ALPHA"],
            priority_eps=cfg["PRIORITY_EPS"],
        )
    else:
        replay = UniformReplayBuffer(cfg["BUFFER_SIZE"])

    nstep = NStepTransitionAccumulator(cfg["N_STEP"], cfg["GAMMA"])

    episode_train_rewards = []
    episode_true_scores = []
    episode_lengths = []
    episode_max_tiles = []
    loss_history = []
    probe_records = []
    forced_abort_episodes = []

    global_step = 0
    start_episode = 1
    resume_info = None

    candidate_resume = resume_from
    if candidate_resume is None and cfg.get("RESUME_PATH"):
        candidate_resume = cfg["RESUME_PATH"]
    if candidate_resume is None and cfg.get("AUTO_RESUME", False):
        candidate_resume = "latest"

    if candidate_resume:
        if str(candidate_resume).lower() == "latest":
            ckpt_path = latest_checkpoint_path(cfg, train_seed=train_seed)
        else:
            ckpt_path = Path(candidate_resume)
        if ckpt_path is not None and Path(ckpt_path).exists():
            payload = load_training_checkpoint(ckpt_path, map_location=DEVICE)
            q_net.load_state_dict(payload["model_state_dict"])
            target_net.load_state_dict(payload["target_state_dict"])
            optimizer.load_state_dict(payload["optimizer_state_dict"])
            target_net.eval()

            if payload.get("replay_saved", False) and ("replay" in payload):
                replay = payload["replay"]
            else:
                print("[resume] Lightweight checkpoint found: replay buffer was not saved. Continuing with an empty replay buffer.")

            episode_train_rewards = list(payload.get("episode_returns", []))
            episode_true_scores = list(payload.get("episode_true_scores", []))
            episode_lengths = list(payload.get("episode_lengths", []))
            episode_max_tiles = list(payload.get("episode_max_tiles", []))
            loss_history = list(payload.get("loss_history", []))
            probe_records = list(payload.get("probe_records", []))
            global_step = int(payload.get("global_step", 0))
            start_episode = int(payload.get("episode_completed", 0)) + 1
            _restore_rng_state(payload.get("rng_state"))
            resume_info = {
                "path": str(ckpt_path),
                "episode_completed": int(payload.get("episode_completed", 0)),
                "global_step": global_step,
                "replay_saved": bool(payload.get("replay_saved", False)),
            }
            print(f"[resume] Loaded checkpoint: {ckpt_path}")
            print(f"[resume] Resuming from episode {start_episode} / {cfg['NUM_EPISODES']} (global_step={global_step})")
        elif candidate_resume not in (None, False, "", "latest"):
            print(f"[resume] Requested checkpoint not found: {candidate_resume}")

    training_started_at = time.time()
    stop_reason = "completed"

    def save_progress(episode_completed, reason, tag="episode", include_replay=None):
        payload = make_training_checkpoint(
            cfg=cfg,
            train_seed=train_seed,
            episode_completed=episode_completed,
            global_step=global_step,
            q_net=q_net,
            target_net=target_net,
            optimizer=optimizer,
            replay=replay,
            episode_train_rewards=episode_train_rewards,
            episode_true_scores=episode_true_scores,
            episode_lengths=episode_lengths,
            episode_max_tiles=episode_max_tiles,
            loss_history=loss_history,
            probe_records=probe_records,
            in_channels=in_channels,
            num_actions=num_actions,
            stop_reason=reason,
            include_replay=include_replay,
        )
        path, latest_path = save_training_checkpoint(payload, cfg, train_seed, episode_completed, tag=tag)
        print(f"[checkpoint] saved: {path}")
        print(f"[checkpoint] latest : {latest_path}")
        return str(path)

    pbar = tqdm(
        range(start_episode, cfg["NUM_EPISODES"] + 1),
        initial=max(0, start_episode - 1),
        total=cfg["NUM_EPISODES"],
        desc=f"Training seed {train_seed}",
    )

    try:
        for episode in pbar:
            obs = env.reset(seed=train_seed + episode)
            nstep.reset()
            done = False
            ep_train_reward = 0.0
            ep_true_score = 0.0
            ep_len = 0
            ep_max_tile = 0
            episode_start = time.time()
            episode_forced_abort = False

            while not done and ep_len < cfg["MAX_STEPS_PER_EPISODE"]:
                elapsed_ep = time.time() - episode_start
                if elapsed_ep > float(cfg.get("EPISODE_ABORT_SECONDS", 1e18)):
                    print(
                        f"[warn] Episode {episode} exceeded EPISODE_ABORT_SECONDS="
                        f"{cfg['EPISODE_ABORT_SECONDS']}s. Aborting this episode and continuing."
                    )
                    episode_forced_abort = True
                    forced_abort_episodes.append({
                        "episode": int(episode),
                        "elapsed_seconds": float(elapsed_ep),
                        "steps": int(ep_len),
                    })
                    break

                legal = env.legal_actions()
                legal_mask = make_legal_mask(num_actions, legal)
                eps = epsilon_by_step(global_step, cfg)
                action = masked_greedy_action(q_net, obs, legal, num_actions, epsilon=eps, device=DEVICE)

                next_obs, reward, done, info = env.step(action)
                next_legal = env.legal_actions() if not done else []
                next_legal_mask = make_legal_mask(num_actions, next_legal)

                ready_transitions = nstep.add(
                    obs, action, reward, next_obs, done, legal_mask, next_legal_mask
                )
                for tr in ready_transitions:
                    replay.add(*tr)

                obs = next_obs
                ep_train_reward += float(reward)
                ep_true_score += float(info["delta_return"])
                ep_len += 1
                ep_max_tile = max(ep_max_tile, int(info["max_tile"]))
                global_step += 1

                if len(replay) >= max(cfg["BATCH_SIZE"], cfg["LEARN_START"]) and global_step % cfg["LEARN_EVERY"] == 0:
                    beta = per_beta_by_step(global_step, cfg) if cfg["USE_PRIORITIZED_REPLAY"] else 1.0
                    batch, indices, weights = replay.sample(cfg["BATCH_SIZE"], beta=beta)
                    loss, priorities = dqn_update(batch, weights, q_net, target_net, optimizer, cfg)
                    replay.update_priorities(indices, priorities)
                    loss_history.append(loss)

                if global_step % cfg["TARGET_SYNC_EVERY"] == 0:
                    target_net.load_state_dict(q_net.state_dict())

            episode_elapsed = time.time() - episode_start
            if episode_elapsed > float(cfg.get("EPISODE_WARN_SECONDS", 1e18)):
                print(
                    f"[warn] Episode {episode} took {episode_elapsed:.1f}s for {ep_len} steps. "
                    "This may indicate a slow runtime or a near-stuck episode."
                )

            episode_train_rewards.append(float(ep_train_reward))
            episode_true_scores.append(float(ep_true_score))
            episode_lengths.append(int(ep_len))
            episode_max_tiles.append(int(ep_max_tile))

            if cfg["PROBE_EVERY_EPISODES"] and episode % cfg["PROBE_EVERY_EPISODES"] == 0:
                probe_summary, _ = evaluate_greedy_policy(
                    q_net,
                    seeds=cfg["PROBE_SEEDS"],
                    cfg=cfg,
                    reward_mode=cfg["REWARD_MODE"],
                    print_each_seed=False,
                )
                probe_records.append({
                    "episode": episode,
                    "mean_greedy_return": probe_summary["mean_greedy_return"],
                    "mean_train_reward": probe_summary["mean_train_reward"],
                    "mean_max_tile": probe_summary["mean_max_tile"],
                    "fraction_illegal_actions_avoided": probe_summary["fraction_illegal_actions_avoided"],
                })

            recent_scores = episode_true_scores[-20:] if episode_true_scores else [0.0]
            recent_max_tiles = episode_max_tiles[-20:] if episode_max_tiles else [0.0]
            pbar.set_postfix({
                "eps": f"{epsilon_by_step(global_step, cfg):.3f}",
                "score_ma20": f"{np.mean(recent_scores):.1f}",
                "tile_ma20": f"{np.mean(recent_max_tiles):.1f}",
                "replay": len(replay),
            })

            if cfg.get("PRINT_HEARTBEAT", True) and episode % int(cfg.get("HEARTBEAT_EVERY_EPISODES", 10)) == 0:
                total_elapsed = time.time() - training_started_at
                print(
                    f"[heartbeat] ep={episode}/{cfg['NUM_EPISODES']} | global_step={global_step} | "
                    f"eps={epsilon_by_step(global_step, cfg):.4f} | replay={len(replay)} | "
                    f"score_ma20={np.mean(recent_scores):.1f} | tile_ma20={np.mean(recent_max_tiles):.1f} | "
                    f"last_ep_s={episode_elapsed:.1f} | total_elapsed_min={total_elapsed/60.0:.1f}",
                    flush=True,
                )

            if episode % int(cfg.get("CHECKPOINT_EVERY_EPISODES", 50)) == 0:
                include_replay = bool(cfg.get("SAVE_REPLAY_IN_CHECKPOINT", False))
                save_progress(episode, reason="periodic", tag="episode", include_replay=include_replay)

        if len(episode_true_scores) > 0:
            save_progress(len(episode_true_scores), reason="completed", tag="completed", include_replay=False)

    except KeyboardInterrupt:
        stop_reason = "keyboard_interrupt"
        completed = len(episode_true_scores)
        print(f"[interrupt] Training stopped manually at episode {completed}.")
        if completed > 0:
            save_progress(completed, reason=stop_reason, tag="interrupted", include_replay=False)
        else:
            print("[interrupt] No completed episode yet, so there is nothing useful to checkpoint.")

    except Exception as exc:
        stop_reason = f"exception: {type(exc).__name__}: {exc}"
        completed = len(episode_true_scores)
        print("[error] Training failed, saving checkpoint before exiting.")
        print(traceback.format_exc())
        if completed > 0:
            save_progress(completed, reason=stop_reason, tag="error", include_replay=False)
        raise

    final_summary, rollout_trace = evaluate_greedy_policy(
        q_net,
        seeds=cfg["EVAL_SEEDS"],
        cfg=cfg,
        reward_mode=cfg["REWARD_MODE"],
        render_rollout_seed=cfg["EVAL_SEEDS"][-1],
        print_each_seed=cfg.get("PRINT_EACH_EVAL_SEED", False),
    )

    latest_ckpt = latest_checkpoint_path(cfg, train_seed=train_seed)

    return {
        "train_seed": train_seed,
        "q_net": q_net,
        "target_net": target_net,
        "config": copy.deepcopy(cfg),
        "in_channels": in_channels,
        "num_actions": num_actions,
        "episode_returns": episode_train_rewards,   # training reward under wrapper
        "episode_true_scores": episode_true_scores, # true game score during training
        "episode_lengths": episode_lengths,
        "episode_max_tiles": episode_max_tiles,
        "loss_history": loss_history,
        "probe_df": pd.DataFrame(probe_records),
        "final_eval": final_summary,
        "rollout_trace": rollout_trace,
        "resume_info": resume_info,
        "latest_checkpoint_path": str(latest_ckpt) if latest_ckpt is not None else None,
        "stopped_early": stop_reason != "completed",
        "stop_reason": stop_reason,
        "forced_abort_df": pd.DataFrame(forced_abort_episodes),
    }



## 6c. Nếu train bị đứng giữa chừng

Cách dùng trong notebook này:
- cell train sẽ **tự resume từ checkpoint gần nhất**
- nếu bạn bấm stop / runtime bị ngắt, chạy lại các cell setup rồi chạy lại cell train
- checkpoint nhẹ được lưu định kỳ trong `CHECKPOINT_DIR`
- `STATUS_JSON` ghi lại checkpoint mới nhất và episode đã hoàn thành

Mẹo:
- nếu train chậm bất thường, giảm `NUM_EPISODES`, giảm `EVAL_SEEDS`, hoặc tăng `CHECKPOINT_EVERY_EPISODES`
- nếu muốn resume từ file cụ thể: đặt `CONFIG["RESUME_PATH"] = "<path_to_ckpt>"`


In [ ]:

# Start training (auto-resume from latest checkpoint by default)
run = train_one_seed(CONFIG, resume_from="latest")
print("Done. Final mean greedy return (true game score):", run["final_eval"]["mean_greedy_return"])
print("Final mean max tile:", run["final_eval"]["mean_max_tile"])
print("Fraction illegal actions avoided:", run["final_eval"]["fraction_illegal_actions_avoided"])
print("Latest checkpoint:", run.get("latest_checkpoint_path"))
if run.get("resume_info"):
    print("Resumed from:", run["resume_info"])
if run.get("stopped_early"):
    print("Training ended early:", run.get("stop_reason"))


Training seed 7:   0%|          | 9/3000 [00:01<06:57,  7.16it/s, eps=0.989, score_ma20=1037.2, tile_ma20=105.6, replay=1148]

[heartbeat] ep=10/3000 | global_step=1148 | eps=0.9886 | replay=1148 | score_ma20=1037.2 | tile_ma20=105.6 | last_ep_s=0.1 | total_elapsed_min=0.0


Training seed 7:   1%|          | 19/3000 [00:03<07:52,  6.30it/s, eps=0.976, score_ma20=1158.8, tile_ma20=116.8, replay=2460]

[heartbeat] ep=20/3000 | global_step=2460 | eps=0.9756 | replay=2460 | score_ma20=1158.8 | tile_ma20=116.8 | last_ep_s=0.2 | total_elapsed_min=0.1


Training seed 7:   1%|          | 29/3000 [00:04<06:24,  7.73it/s, eps=0.965, score_ma20=1121.6, tile_ma20=115.2, replay=3526]

...

Training seed 7:   2%|▏         | 69/3000 [00:28<52:30,  1.07s/it, eps=0.923, score_ma20=1047.6, tile_ma20=102.4, replay=7817]  

[heartbeat] ep=70/3000 | global_step=7817 | eps=0.9226 | replay=7817 | score_ma20=1047.6 | tile_ma20=102.4 | last_ep_s=0.9 | total_elapsed_min=0.5


Training seed 7:   3%|▎         | 79/3000 [00:37<37:53,  1.28it/s, eps=0.912, score_ma20=998.6, tile_ma20=94.4, replay=8843]  

[heartbeat] ep=80/3000 | global_step=8843 | eps=0.9125 | replay=8843 | score_ma20=998.6 | tile_ma20=94.4 | last_ep_s=0.6 | total_elapsed_min=0.6


Training seed 7:   3%|▎         | 86/3000 [00:44<53:43,  1.11s/it, eps=0.905, score_ma20=1029.2, tile_ma20=99.2, replay=9629] 

In [ ]:

status_path = Path(CONFIG["STATUS_JSON"])
if status_path.exists():
    print("Status JSON:", status_path)
    print(status_path.read_text())

if not run["forced_abort_df"].empty:
    print("Forced-abort episodes detected:")
    display(run["forced_abort_df"])


## 7. Visualize training and probe metrics

`episode_returns` là **training reward** theo wrapper đã chọn.  
`episode_true_scores` là **game score thật** trong khi train.

In [ ]:
def moving_average(x, w=20):
    x = np.asarray(x, dtype=np.float32)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w, dtype=np.float32) / w, mode="valid")


plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.plot(run["episode_true_scores"], alpha=0.35, label="true game score")
ma = moving_average(run["episode_true_scores"], 20)
plt.plot(range(len(ma)), ma, label="moving avg (20)")
plt.title("Training true game score")
plt.xlabel("Episode")
plt.ylabel("Score")
plt.legend()

plt.subplot(1, 4, 2)
plt.plot(run["episode_lengths"], alpha=0.5)
plt.title("Episode length")
plt.xlabel("Episode")
plt.ylabel("Steps")

plt.subplot(1, 4, 3)
plt.plot(run["episode_max_tiles"], alpha=0.5)
plt.title("Max tile per episode")
plt.xlabel("Episode")
plt.ylabel("Tile")

plt.subplot(1, 4, 4)
plt.plot(run["loss_history"], alpha=0.7)
plt.title("Huber TD loss")
plt.xlabel("Update step")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

if not run["probe_df"].empty:
    probe_df = run["probe_df"]
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.plot(probe_df["episode"], probe_df["mean_greedy_return"], marker="o")
    plt.title("Probe mean greedy return")
    plt.xlabel("Episode")
    plt.ylabel("True score")

    plt.subplot(1, 3, 2)
    plt.plot(probe_df["episode"], probe_df["mean_max_tile"], marker="o")
    plt.title("Probe mean max tile")
    plt.xlabel("Episode")
    plt.ylabel("Tile")

    plt.subplot(1, 3, 3)
    plt.plot(probe_df["episode"], probe_df["fraction_illegal_actions_avoided"], marker="o")
    plt.title("Probe fraction illegal avoided")
    plt.xlabel("Episode")
    plt.ylabel("Fraction")

    plt.tight_layout()
    plt.show()

## 8. Final multi-seed evaluation

Đây là cell quan trọng nhất cho báo cáo vì nó trả đúng 3 metric thầy yêu cầu.

Điểm mới:
- in ra **score từng seed**
- sau đó mới in **mean / std / median**
- `mean greedy return` là **true game score**

In [ ]:
final_eval = run["final_eval"]

print("Per-seed evaluation results:")
for row in final_eval["per_seed_results"].itertuples(index=False):
    print(
        f"  seed={row.seed} | score={row.greedy_return:.2f} | "
        f"max_tile={row.max_tile} | steps={row.steps} | train_reward={row.train_reward_sum:.2f}"
    )

print("\nAggregate metrics:")
print("Mean greedy return:", final_eval["mean_greedy_return"])
print("Std greedy return:", final_eval["std_greedy_return"])
print("Median greedy return:", final_eval["median_greedy_return"])
print("Mean training reward:", final_eval["mean_train_reward"])
print("Mean max tile:", final_eval["mean_max_tile"])
print("Max tile distribution:", final_eval["max_tile_distribution"])
print("Fraction of illegal actions avoided:", final_eval["fraction_illegal_actions_avoided"])
print("Number of eval seeds:", final_eval["num_eval_seeds"])

display(final_eval["table"].head(15))

In [ ]:
# A compact summary table for reporting
report_df = pd.DataFrame([{
    "train_seed": run["train_seed"],
    "reward_mode": run["config"]["REWARD_MODE"],
    "n_step": run["config"]["N_STEP"],
    "use_prioritized_replay": run["config"]["USE_PRIORITIZED_REPLAY"],
    "num_eval_seeds": final_eval["num_eval_seeds"],
    "mean_greedy_return": final_eval["mean_greedy_return"],
    "std_greedy_return": final_eval["std_greedy_return"],
    "median_greedy_return": final_eval["median_greedy_return"],
    "mean_train_reward": final_eval["mean_train_reward"],
    "mean_max_tile": final_eval["mean_max_tile"],
    "fraction_illegal_actions_avoided": final_eval["fraction_illegal_actions_avoided"],
}])
report_df


## 9. Inspect one greedy rollout


In [ ]:

rollout = run["rollout_trace"] or []
print("Rollout length:", len(rollout))
if rollout:
    print("Final board:")
    print(rollout[-1]["board"])

n_show = min(5, len(rollout))
for i, step_info in enumerate(rollout[-n_show:], start=max(1, len(rollout) - n_show + 1)):
    print("=" * 70)
    print(
        f"Step {i} | action={step_info['action']} | raw_argmax={step_info['raw_argmax']} | "
        f"raw_argmax_is_legal={step_info['raw_argmax_is_legal']} | reward={step_info['reward']:.2f}"
    )
    print(step_info["board"])



## 10. Optional ablations

Nếu muốn viết báo cáo kỹ hơn, bạn có thể chạy các ablation sau:

1. `REWARD_MODE = "delta_return"`
2. `REWARD_MODE = "state_rewards"`
3. `REWARD_MODE = "shaped"`
4. `USE_DUELING = False`
5. giảm / tăng `EPS_DECAY_STEPS`, `LR`, `NUM_EPISODES`

Cell dưới đây giúp bạn chạy nhanh vài biến thể bằng cách copy config gốc.


In [ ]:

# Example ablation configs (disabled by default because training is expensive)
ABLATIONS = [
    {"name": "delta_return_baseline", "REWARD_MODE": "delta_return", "USE_DUELING": True},
    {"name": "state_rewards", "REWARD_MODE": "state_rewards", "USE_DUELING": True},
    {"name": "shaped_reward", "REWARD_MODE": "shaped", "USE_DUELING": True},
    {"name": "no_dueling", "REWARD_MODE": "delta_return", "USE_DUELING": False},
]

# To run ablations, uncomment below.
# ablation_rows = []
# for variant in ABLATIONS:
#     cfg = copy.deepcopy(CONFIG)
#     cfg.update({k: v for k, v in variant.items() if k != "name"})
#     result = train_one_seed(cfg, train_seed=cfg["TRAIN_SEED"])
#     ablation_rows.append({
#         "name": variant["name"],
#         "mean_greedy_return": result["final_eval"]["mean_greedy_return"],
#         "mean_max_tile": result["final_eval"]["mean_max_tile"],
#         "fraction_illegal_actions_avoided": result["final_eval"]["fraction_illegal_actions_avoided"],
#     })
# pd.DataFrame(ablation_rows).sort_values("mean_greedy_return", ascending=False)



## 11. Save checkpoint


In [ ]:

checkpoint = {
    "model_state_dict": run["q_net"].state_dict(),
    "target_state_dict": run["target_net"].state_dict(),
    "config": run["config"],
    "in_channels": run["in_channels"],
    "num_actions": run["num_actions"],
    "episode_returns": run["episode_returns"],
    "episode_true_scores": run["episode_true_scores"],
    "episode_lengths": run["episode_lengths"],
    "episode_max_tiles": run["episode_max_tiles"],
    "loss_history": run["loss_history"],
    "final_eval": {
        k: v for k, v in run["final_eval"].items() if k not in {"table", "per_seed_results"}
    },
    "latest_checkpoint_path": run.get("latest_checkpoint_path"),
}

torch.save(checkpoint, CONFIG["SAVE_PATH"])
print("Saved final model checkpoint to:", CONFIG["SAVE_PATH"])
print("Latest resumable training checkpoint:", run.get("latest_checkpoint_path"))


## 12. What this notebook now implements from the proposal

Notebook này đã gộp các ý đã đề xuất:

- improved observation wrapper (one-hot + scalar feature maps)
- CNN + dueling heads
- Double DQN
- Huber loss
- **stronger reward shaping for 2048**
- **n-step targets**
- **prioritized replay**
- final multi-seed evaluation
- per-seed score printing
- illegal-action metric based on raw argmax before masking

### Do you need to train longer?

Thường là **có**, đặc biệt nếu:
- bạn muốn 512 xuất hiện thường xuyên hơn
- bạn muốn mean greedy return ổn định hơn
- bạn tăng độ phức tạp của model / replay / reward shaping

Gợi ý thực tế:
- **debug / smoke test**: 800-1500 episodes
- **bản báo cáo nghiêm túc**: 3000-5000 episodes
- **nếu còn budget thời gian**: 5000-8000 episodes + thử nhiều train seeds